**Putusan per-section structured-extractor SFT (Stage 1)** — `Qwen/Qwen3.5-4B`, language-only. Given a full reconstructed decision, it emits JSON for the requested 1–2 sections. This profile uses 4-bit QLoRA and Unsloth's long-context stack on a single **A100 40GB** with a 49,152-token cap.
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

In [1]:
!nvidia-smi

Sun Jul 12 06:58:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Installation

In [ ]:
%%capture
import os, importlib.util
# Reduce allocator fragmentation before torch/Unsloth initialize CUDA.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [ ]:
from unsloth import FastLanguageModel  # language-only SFT
import torch

# This notebook is intentionally profiled for the observed Colab A100-SXM4-40GB.
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA runtime with an NVIDIA A100 is required.")
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if "A100" not in gpu_name or gpu_vram_gb < 39:
    raise RuntimeError(
        f"Expected an A100 with at least 39 GiB, found {gpu_name} ({gpu_vram_gb:.2f} GiB)."
    )
print(f"Long-context profile: {gpu_name}, {gpu_vram_gb:.2f} GiB, BF16={torch.cuda.is_bf16_supported()}")

# The saved training distribution has p95=48,804 formatted tokens. 49,152 is the
# next multiple of 256 and remains below Qwen3.5's native 262,144-token limit,
# so no static YaRN/RoPE scaling is applied.
max_seq_length = 49_152

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-4B",
    max_seq_length = max_seq_length,
    dtype = torch.bfloat16,           # A100-native compute dtype for the 4-bit base
    load_in_4bit = True,              # user-selected QLoRA memory/quality tradeoff
    load_in_8bit = False,
    full_finetuning = False,
    text_only = True,                 # do not load the unused vision tower
    use_gradient_checkpointing = "unsloth",  # CPU activation offload + recomputation
    unsloth_tiled_mlp = True,         # tile each MLP along the sequence dimension
)

# Verify that Tiled MLP reached every language block instead of trusting the flag silently.
tiled_mlp_modules = [
    name for name, module in model.named_modules()
    if name.lower().endswith(".mlp") and hasattr(module, "_original_forward")
]
text_config = getattr(model.config, "text_config", model.config)
expected_mlp_modules = int(text_config.num_hidden_layers)
assert len(tiled_mlp_modules) == expected_mlp_modules, (
    f"Tiled MLP patched {len(tiled_mlp_modules)}/{expected_mlp_modules} language blocks"
)
print(
    f"Tiled MLP active on {len(tiled_mlp_modules)} blocks; "
    f"Arctic tile size = hidden_size = {text_config.hidden_size:,} tokens."
)

We now add LoRA adapters so we only train ~1% of parameters. This is a **language-only** extractor (the vision path is removed), so we target the attention + MLP projection modules directly.

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model as peft_get_peft_model
from collections import Counter

# Two Unsloth bugs on this build for hybrid Qwen3.5:
#  1. from_pretrained auto-attaches LoRA with a q/k/v/o+MLP-only default, leaving
#     the 24 linear-attention layers frozen.
#  2. get_peft_model intersects even EXPLICIT target_modules with its
#     finetune_attention filter, which only knows q/k/v/o names — so the
#     linear-attention projections (in_proj_qkv / in_proj_z / in_proj_a /
#     in_proj_b / out_proj) are silently dropped again.
# Workaround: unload any incomplete adapters, then apply PEFT's LoraConfig
# directly, which honors the target list verbatim. The 500k-context stack
# (tiled MLP, Unsloth gradient checkpointing, chunked CE) lives on the base
# model from from_pretrained, not in get_peft_model — verified at the bottom.

def lora_target_names(m):
    return Counter(name.split(".")[-4] for name, _ in m.named_parameters() if "lora_A" in name)

def covers_linear_attention(names):
    return any(n.startswith("in_proj") or n == "out_proj" for n in names)

if isinstance(model, PeftModel) and not covers_linear_attention(lora_target_names(model)):
    print("Attached adapters miss the linear-attention projections — unloading them.")
    model = model.unload()  # drops the incomplete (untrained) adapters, keeps the 4-bit base

if not isinstance(model, PeftModel):
    # Enumerate every linear projection in the language blocks so both attention
    # flavors and the MLPs are covered; skip lm_head, embeddings, and any vision path.
    import torch.nn as nn
    target_modules = sorted({
        name.split(".")[-1]
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)  # bnb Linear4bit subclasses nn.Linear
        and not any(skip in name for skip in ("lm_head", "embed", "visual", "vision"))
    })
    print(f"LoRA target modules: {target_modules}")
    lora_config = LoraConfig(
        r = 32,
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        target_modules = target_modules,
        task_type = "CAUSAL_LM",
    )
    model = peft_get_peft_model(model, lora_config)
    # Gradient checkpointing (use_gradient_checkpointing="unsloth") was enabled at
    # from_pretrained; frozen embeddings need grad-capable inputs under it.
    model.enable_input_require_grads()
    model.print_trainable_parameters()

# Verify config and coverage on whichever path attached the adapters.
peft_cfg = model.peft_config["default"]
print(f"r={peft_cfg.r}  lora_alpha={peft_cfg.lora_alpha}  lora_dropout={peft_cfg.lora_dropout}")
adapter_targets = lora_target_names(model)
print(adapter_targets)
assert peft_cfg.r == 32 and peft_cfg.lora_alpha == 32, (
    f"Adapter config differs from the intended r=32/alpha=32 profile: "
    f"r={peft_cfg.r}, alpha={peft_cfg.lora_alpha}"
)
assert covers_linear_attention(adapter_targets), (
    "Qwen3.5 linear-attention layers did not receive LoRA adapters"
)

# The 500k-context stack (tiled MLP + Unsloth gradient checkpointing + chunked CE,
# https://unsloth.ai/docs/blog/500k-context-length-fine-tuning) is patched into the
# base model at from_pretrained, not by get_peft_model — re-verify it survived
# unload() and the plain-PEFT wrap. Tiled MLP's patched forward calls
# self.gate_proj/up_proj/down_proj by attribute, so it runs through the LoRA
# wrappers PEFT injected.
tiled_after_peft = [
    name for name, module in model.named_modules()
    if name.lower().endswith(".mlp") and hasattr(module, "_original_forward")
]
assert len(tiled_after_peft) == expected_mlp_modules, (
    f"Tiled MLP patch lost after PEFT wrap: {len(tiled_after_peft)}/{expected_mlp_modules} blocks"
)
base_lm = model.get_base_model()
assert getattr(base_lm, "is_gradient_checkpointing", False) or getattr(
    base_lm, "gradient_checkpointing", False
), "Gradient checkpointing is no longer enabled on the base model"
print(f"Long-context stack intact: tiled MLP on {len(tiled_after_peft)} blocks, gradient checkpointing on.")

# Unsloth's patched generate iterates self.config.architectures, but the
# text_only=True load leaves it None — backfill so the inference cells work.
if getattr(base_lm.config, "architectures", None) is None:
    base_lm.config.architectures = [type(base_lm).__name__]
    print(f"Backfilled config.architectures = {base_lm.config.architectures}")

<a name="Data"></a>
### Data Prep — per-section extraction (`sft_sections`)

We fine-tune a **structured section extractor**: given a **full** putusan (court decision) body, emit one JSON object containing only the **1-2 requested sections** — matching inference, where a user uploads a whole decision and asks for one or two sections.

The dataset lives in [`Haeryz/putusan-structured-extraction`](https://huggingface.co/datasets/Haeryz/putusan-structured-extraction), config **`sft_sections`** (built by `notebooks/build_sections_dataset.py` from the legacy whole-doc `sft` config — same document-disjoint splits, zero leakage). Each document yields six examples with deterministic, difficulty-weighted section sampling: `ahli` (67% empty in gold) every doc, `penangkapan`/`surat` alternating, two of the five long-body sections rotating, one medium section rotating, and one rotating pair of trivial identity/date sections. ~16% of examples ask for a section that is empty in that document, so the model learns to answer with an empty list instead of hallucinating.

Each row carries a ready `messages` column (system prompt naming the requested sections, full putusan body as `user`, gold per-section JSON as `assistant`), so we just load the config below — no local build step needed.

In [3]:
# Load the per-section SFT config from the Hub (configs: sft / sft_sections / grpo / rag).
from datasets import load_dataset

DATASET_REPO = "Haeryz/putusan-structured-extraction"
dataset      = load_dataset(DATASET_REPO, "sft_sections", split = "train")
eval_dataset = load_dataset(DATASET_REPO, "sft_sections", split = "validation")

Let's look at the dataset — each row is a chat with a `system` extraction instruction, the reconstructed putusan body as `user`, and the gold 31-section JSON as `assistant`.

In [4]:
dataset

Dataset({
    features: ['id', 'parent_id', 'corpus', 'annotator_model', 'source_file', 'source_sha256', 'extraction_method', 'purpose', 'split', 'split_seed', 'requested_sections', 'n_requested', 'n_empty_requested', 'input_text', 'target_json', 'messages', 'prompt', 'answer', 'n_input_chars', 'n_target_chars'],
    num_rows: 14766
})

In [5]:
# Peek at the assistant target (gold JSON) for the first example.
print(dataset[0]["messages"][2]["content"][:1000])

{"sections": {"ahli": []}, "empty_sections": ["ahli"]}


We format each chat into a single `text` string with the model's chat template, so the standard text `SFTTrainer` can tokenize it (no vision collator needed).

In [6]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset      = dataset.map(formatting_prompts_func, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

Measure the fully formatted sequence-length distribution and set `max_length` from the **95th percentile**, capped at the A100-40GB profile's `max_seq_length`.

In [ ]:
import numpy as np, os, wandb
from tqdm.auto import tqdm

# `text_only=True` normally returns an AutoTokenizer; this fallback also supports older
# Unsloth builds that return a multimodal Processor.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# Measuring 14.8k long documents takes minutes — cache the lengths as a W&B Artifact
# so reruns skip straight to the percentiles. Requires being logged in to wandb here
# (the SFTTrainer below reuses this same run via report_to="wandb").
ARTIFACT = "token-lengths-sft_sections-train"
run = wandb.run or wandb.init(project = "huggingface")

token_lengths = None
try:
    art_dir = run.use_artifact(f"{ARTIFACT}:latest").download()
    cached = np.load(os.path.join(art_dir, "token_lengths.npy"))
    if len(cached) == len(dataset):  # invalidate the cache if the dataset changed
        token_lengths = cached
        print(f"Loaded {len(cached)} cached token lengths from W&B artifact {ARTIFACT}:latest")
except Exception as e:
    print(f"No usable cached measurement ({type(e).__name__}) — measuring now")

if token_lengths is None:
    # Batched tokenization: the fast tokenizer parallelizes across a batch,
    # far quicker than calling it once per document.
    texts = dataset["text"]
    token_lengths = np.array([
        len(ids)
        for i in tqdm(range(0, len(texts), 256), desc = "measuring token lengths", unit = "batch")
        for ids in text_tokenizer(texts[i : i + 256], add_special_tokens = False)["input_ids"]
    ])
    np.save("token_lengths.npy", token_lengths)
    art = wandb.Artifact(ARTIFACT, type = "measurement",
                         metadata = {"rows": len(dataset), "dataset": DATASET_REPO, "config": "sft_sections"})
    art.add_file("token_lengths.npy")
    run.log_artifact(art)
    print(f"Measured and cached {len(token_lengths)} token lengths to W&B artifact {ARTIFACT}")

p50, p90, p95 = (int(np.percentile(token_lengths, p)) for p in (50, 90, 95))
# Round p95 up to a multiple of 256; never exceed the A100 profile's context cap.
MAX_LENGTH = int(min(np.ceil(p95 / 256) * 256, max_seq_length))
train_context_coverage = float(np.mean(token_lengths <= MAX_LENGTH))
print(f"token length  p50={p50}  p90={p90}  p95={p95}  max={int(max(token_lengths))}")
print(f"MAX_LENGTH (95th pct, capped at {max_seq_length}) = {MAX_LENGTH}")
print(f"train rows fitting without truncation = {train_context_coverage:.2%}")
assert p95 <= MAX_LENGTH <= max_seq_length, (
    f"MAX_LENGTH={MAX_LENGTH} must cover p95={p95} without exceeding the {max_seq_length} profile cap"
)
assert train_context_coverage >= 0.94, "Dataset drift pushed too many rows beyond the context cap"
tile_count = int(np.ceil(MAX_LENGTH / int(text_config.hidden_size)))
print(f"Tiled MLP will use about {tile_count} sequence tiles at MAX_LENGTH.")

Here is the fully-formatted `text` for the first example (system + user putusan body + assistant JSON):

In [8]:
print(dataset[0]["text"][:2000])

<|im_start|>system
Anda adalah pengekstrak terstruktur putusan pengadilan Indonesia. Diberikan badan teks putusan, keluarkan SATU objek JSON yang hanya berisi bagian yang diminta. Setiap nilai adalah daftar kutipan verbatim (extractive) yang disalin persis dari teks sumber — jangan pernah memparafrasekan, meringkas, atau mengarang. Jika bagian yang diminta tidak ada dalam teks, gunakan daftar kosong dan cantumkan kuncinya di 'empty_sections'. Bagian yang diminta: ahli.<|im_end|>
<|im_start|>user
P U T U S A N

Nomor 2/Pid.Sus-Anak/2026/PN Arm

DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA

Pengadilan Negeri Pengadilan Negeri Airmadidi

Pengadilan Negeri Pengadilan Negeri Airmadidi yang mengadili perkara
pidana  anak  dengan acara  pemeriksaan  biasa  dalam  tingkat  pertama
menjatuhkan putusan sebagai berikut dalam perkara Anak:

ANAK

Perawang

13/12 Mei 2012

Laki-laki

Indonesia

Kota Manado

Budha

Belum bekerja

Anak tidak ditahan;

Setelah  mendengar  pembacaan  tuntutan  pid

Before finetuning, let's see what the base model emits for the first putusan body (system + user only, assistant left blank).

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Unsloth's patched generate iterates self.config.architectures; text_only loading
# leaves it None on some (or all) of the configs in the model tree. Backfill every
# reachable config so the VLM check gets a real list, whichever config it reads.
_arch = [type(model.get_base_model()).__name__]
_patched = 0
for _mod in [model, *(m for _, m in model.named_modules())]:
    _cfg = getattr(_mod, "config", None)
    if _cfg is not None and getattr(_cfg, "architectures", None) is None:
        try:
            _cfg.architectures = _arch
            _patched += 1
        except Exception:
            pass
print(f"Backfilled architectures={_arch} on {_patched} config object(s)")

# Use the inner text tokenizer: the Qwen3VLProcessor's apply_chat_template defaults to
# tokenize=False and returns a str (ignoring return_tensors), so .to("cuda") would fail.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# system + user (drop the gold assistant turn); ask the model to produce the JSON.
prompt_messages = dataset[0]["messages"][:2]
inputs = text_tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                   use_cache = True, temperature = 0.7, min_p = 0.1,
                   pad_token_id = text_tokenizer.eos_token_id)

<a name="Train"></a>
### Train the model

Standard text `SFTTrainer` (no `UnslothVisionDataCollator`). We train on the `text` field for `num_train_epochs = 2` (ORCHESTRATION Stage 1 says 1-3), with `max_length` set to the measured 90th-percentile token length. We then use `train_on_responses_only` so the loss is computed **only on the assistant JSON**, not on the (very long) putusan input. Set `max_steps` for a quick smoke test.

In [ ]:
import os
from trl import SFTTrainer, SFTConfig

# Force Unsloth's fused/chunked cross-entropy in BOTH train and eval forwards.
# Without it, evaluation materializes the full [seq_len, vocab] logits tensor
# (~21 GB bf16 at 48,896 tokens x 248k vocab) and accelerate then upcasts it to
# fp32 (~42 GB) -> the OOM observed at the step-20 eval.
os.environ["UNSLOTH_RETURN_LOGITS"] = "0"

FastLanguageModel.for_training(model)  # Enable for training!

# Keep the tokenizer fallback for compatibility with text-only and processor loaders.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

train_rows_before_mask = len(dataset)
eval_rows_before_mask = len(eval_dataset)

trainer = SFTTrainer(
    model = model,
    tokenizer = text_tokenizer,
    train_dataset = dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,   # required for reliable long-context tiling
        gradient_accumulation_steps = 8,   # preserves effective batch size = 8
        warmup_steps = 5,
        # num_train_epochs = 1,              # 14,766 per-section examples -> ~923 optimizer steps
        max_steps = 100,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",     # For Weights and Biases
        max_length = MAX_LENGTH,
        packing = False,             # keep each court decision isolated
        bf16 = True,
        fp16 = False,
        eval_strategy = "steps",           # evaluate on the val split during training
        eval_steps = 20,                  # val is 1,866 long examples - per-step eval would dominate runtime
        per_device_eval_batch_size = 1,
        prediction_loss_only = True,       # eval keeps only the loss; never gathers/upcasts logits
    ),
)

Authenticate W&B with Colab Secrets or `WANDB_API_KEY`/`wandb.login()` at runtime. Never paste API keys into notebook cells. The previously embedded credential must be rotated.

In [ ]:
# Train only on the assistant response (the JSON), masking the putusan input.
# Qwen3.5 uses ChatML markers.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

train_retention = len(trainer.train_dataset) / train_rows_before_mask
eval_retention = len(trainer.eval_dataset) / eval_rows_before_mask
print(f"response-supervised rows retained: train={train_retention:.2%}, eval={eval_retention:.2%}")
assert train_retention >= 0.94, "Too many training rows lost their assistant response after truncation"

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Run the finetuned extractor on a held-out putusan body. It should emit JSON containing only the requested 1–2 sections. We use low-temperature decoding for stable structured output.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Same backfill as the pre-training inference cell: Unsloth's patched generate
# needs config.architectures to be a list on whichever config it reads.
_arch = [type(model.get_base_model()).__name__]
for _mod in [model, *(m for _, m in model.named_modules())]:
    _cfg = getattr(_mod, "config", None)
    if _cfg is not None and getattr(_cfg, "architectures", None) is None:
        try:
            _cfg.architectures = _arch
        except Exception:
            pass

# Try a held-out test-split example from the Hub repo (per-section config).
test_dataset = load_dataset(DATASET_REPO, "sft_sections", split = "test")
prompt_messages = test_dataset[0]["messages"][:2]  # system + user, drop the gold assistant turn

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
inputs = text_tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048,
                   use_cache = True, temperature = 0.3, min_p = 0.1,
                   pad_token_id = text_tokenizer.eos_token_id)

<a name="Save"></a>
### Saving, loading finetuned models
Save the Stage-1 LoRA adapters as `qwen_extractor_sft_lora` - Stage 2 (GRPO) continues from this. Use `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, not the full model. To save 16-bit merged for serving, scroll down.

In [ ]:
model.save_pretrained("qwen_extractor_sft_lora")  # Local saving (Stage 2 continues from this)
tokenizer.save_pretrained("qwen_extractor_sft_lora")
# model.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_extractor_sft_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        load_in_4bit = True,  # adapters were trained against the 4-bit base
        text_only = True,
    )
    FastLanguageModel.for_inference(model) # Enable for inference!

    prompt_messages = dataset[0]["messages"][:2]
    text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    inputs = text_tokenizer.apply_chat_template(
        prompt_messages, add_generation_prompt = True, return_tensors = "pt",
        return_dict = True,
    ).to("cuda")
    from transformers import TextStreamer
    text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
    _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048,
                       use_cache = True, temperature = 0.3, min_p = 0.1,
                       pad_token_id = text_tokenizer.eos_token_id)

### Saving to float16 for vLLM

We also support saving to `float16` directly for serving (Stage 3). Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit merged (serving-ready extractor)
if False: model.save_pretrained_merged("qwen_extractor_sft_merged", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/qwen_extractor_sft_merged", tokenizer, token = "YOUR_HF_TOKEN")